In [1]:
import pandas as pd
from pathlib import Path
from functools import reduce

from parreg.process_config import load_and_validate_config
from parreg import utils

In [2]:
config_file = Path('/home/yuqiong.liu/work/Gitlab/ngen-regionalization/configs/config.yaml')
if not config_file.exists():
    raise FileNotFoundError(config_file)

config = load_and_validate_config(config_file)


Exception: Error loading YAML file: to_pandas() got an unexpected keyword argument 'nrows'

In [ ]:
print(config.general.hydrofabric_file)

In [ ]:
datasets = config.general.attr_dataset_list
out1 = config.output.attr_data_final
out2 = config.output.config_final

for vpu in config.general.vpu_list:
    df_attrs_all = []
    for dataset_name in datasets:

        print(f'vpu={vpu}, dataset={dataset_name}')
        dataset = getattr(config.attr_datasets, dataset_name)
        df_attrs = dataset.get_attr_data()
        df_attrs = df_attrs.rename(columns=lambda x: x if x == "divide_id" else f"{dataset_name}_{x}")
        df_attrs_all.append(df_attrs)

    # Merge all attribute data frames column-wise, based on divide_id
    df_attrs_all= reduce(
        lambda left, right: pd.merge(left, right, on='divide_id', how='outer'),
        df_attrs_all
    )

    # check percentage of missing data
    #print(df_attrs_all.isna().mean()*100)

    if out1.save:
        if not Path(out1.path).is_dir():
            raise ValueError(f'')
        utils.save_data(df_attrs_all, Path(out1.path, 'attr_' + config.general.domain + '_vpu' + vpu + '.' + out1.format))

if out2.save:
    if Path(out2.path).is_file():
        utils.save_data(config, Path(out2.path))
    elif Path(out2.path).is_dir():                
        utils.save_data(config, Path(out2.path, 'config_final.' + out2.format))
    

In [ ]:
import geopandas as gpd

gdf1 = gpd.read_file('/home/yuqiong.liu/work/data/gpkg_v2.2/vpu_divides/vpu_01.gpkg')
print(gdf1)

In [ ]:
print(gdf1.columns)

In [ ]:
# original NWMv3 cal/val stats 
domain = 'conus'
df1 = pd.read_csv('/home/yuqiong.liu/work/data/ngen_reg/donor_stats/stats_calib_valid_nwmv3_original_' + domain + '.csv')
df1 = df1[df1['simulation']=='calibrated']

# list of gages for NWMv3 cal/val
gages = pd.read_csv('/home/yuqiong.liu/work/data/ngen_reg/gages_nwm4_calib_all.csv')['gage_id'].unique()

# Identify unique groups in df1 (stats)
group_keys = df1[['jobID', 'domainID']].drop_duplicates().reset_index(drop=True)

# Sample groups to match number of IDs
sampled_groups = group_keys.sample(n=len(gages), random_state=42).reset_index(drop=True)
sampled_groups['gage_id'] = gages

# Merge and filter
df_filtered = df1.merge(sampled_groups, on=['jobID', 'domainID'], how='inner')

# drop the extra columns
df_filtered = df_filtered.drop(columns=['jobID', 'domainID', 'simulation'])

# move gage_id to the front
df_filtered = df_filtered[['gage_id'] + [col for col in df_filtered.columns if col != 'gage_id']]

# save the filtered dataframe
df_filtered.to_csv('/home/yuqiong.liu/work/data/ngen_reg/donor_stats/stats_calib_valid_' + domain + '.csv', index=False)

In [ ]:
print(df_filtered.bias.describe())

In [ ]:

# Step 1: Group df by col1 and col2
group_keys = df1.groupby(['jobID', 'domainID']).ngroup()  # get group numbers
print(group_keys)


In [ ]:

# Step 2: Map group numbers to sampled IDs
group_to_id = dict(zip(sorted(group_keys.unique()), id_pool))

# Step 3: Assign the sampled IDs as a new column
df['group_id'] = group_keys.map(group_to_id)

print(df)

In [ ]:
file1 = "{base_dir}/myfile_{domain}.csv"
variables = {"base_dir": "/tmp", "domain": "conus"}
file2 = file1.format(**variables)  # This will replace {base_dir} with "/tmp"
print(file2)